# Rank IC objective validation v1

目标：把 `knowledge_ingest` 中的 LambdaRankIC / Rank IC 外部知识候选，先验证为 `src/objective` 的 research 证据。这里不复制论文全文、不 vendoring 代码，只验证现有 `evaluate_prediction_objectives` 能表达 Rank IC objective 与 MSE objective 的差异。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.objective import evaluate_prediction_objectives

EXPERIMENT = "rank_ic_objective_validation_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_ID = "arxiv:2605.00501"
SOURCE_URL = "https://arxiv.org/abs/2605.00501"


In [2]:
dates = pd.date_range("2026-01-01", periods=4, freq="D")
base_targets = [0.030, 0.020, -0.010]
rows = []
for index, date in enumerate(dates):
    drift = index * 0.001
    targets = [base_targets[0] + drift, base_targets[1] - drift / 2, base_targets[2] - drift]
    # rank_ic_aligned_signal preserves the cross-sectional order but is not calibrated for MSE.
    rank_predictions = [0.30, 0.20, -0.10]
    # mse_aligned_signal is close in magnitude but flips the top-two order on each date.
    mse_predictions = [targets[1] + 0.001, targets[0] - 0.008, targets[2] + 0.001]
    for instrument, target, rank_pred, mse_pred in zip(
        ["SOXL", "SOXX", "QQQ"], targets, rank_predictions, mse_predictions, strict=True
    ):
        rows.append(
            {
                "date": date,
                "instrument": instrument,
                "target_return": target,
                "rank_ic_aligned_signal": rank_pred,
                "mse_aligned_signal": mse_pred,
            }
        )
frame = pd.DataFrame(rows)

scores = {
    model_name: evaluate_prediction_objectives(
        frame.rename(columns={model_name: "prediction"}),
        prediction_col="prediction",
        target_col="target_return",
        group_col="date",
    )
    for model_name in ["rank_ic_aligned_signal", "mse_aligned_signal"]
}
score_rows = [dict(model=model_name, **payload) for model_name, payload in scores.items()]
pd.DataFrame(score_rows).to_csv(CARD / "objective_scores.csv", index=False, lineterminator="\n")
(CARD / "objective_scores.json").write_text(json.dumps(scores, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
pd.DataFrame(rows).to_csv(CARD / "rank_ic_validation_frame.csv", index=False, lineterminator="\n")
scores


{'rank_ic_aligned_signal': {'schema_version': 'objective.evaluation.v1',
  'row_count': 12,
  'mse': 0.03753262500000001,
  'classification_accuracy': 1.0,
  'ic': 0.9978658093634377,
  'rank_ic': 1.0,
  'rank_group_count': 4},
 'mse_aligned_signal': {'schema_version': 'objective.evaluation.v1',
  'row_count': 12,
  'mse': 5.0416666666666675e-05,
  'classification_accuracy': 1.0,
  'ic': 0.9302709481805915,
  'rank_ic': 0.5,
  'rank_group_count': 4}}

In [3]:
source_provenance = {
    "schema_version": "knowledge.source_provenance.v1",
    "source_id": SOURCE_ID,
    "source_type": "arxiv",
    "title": "LambdaRankIC: Directly Optimizing Rank IC for Financial Prediction",
    "url": SOURCE_URL,
    "summary": "Rank IC objective reference for reducing mismatch between regression losses, generic ranking objectives, and finance evaluation metrics.",
    "retrieved_at": "2026-06-19",
    "access_status": "metadata_accessible",
    "source_used_as": "hypothesis_only",
    "license_note": "只保存 arXiv URL、自有摘要和验证假设；未复制论文全文；未 vendoring 第三方代码。",
}
(CARD / "source_provenance.json").write_text(
    json.dumps(source_provenance, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
(CARD / "license_review.md").write_text(
    "# License Review\n\n"
    "## 中文结论\n\n"
    "- 来源：arXiv metadata URL 和自有摘要。\n"
    "- 未复制论文全文，未保存未授权正文，未 vendoring 论文或第三方实现代码。\n"
    "- 本 notebook 只验证 `src.objective.evaluate_prediction_objectives` 的 Rank IC 表达能力。\n"
    "- 后续如要实现 LambdaRankIC loss，需要单独复核论文和代码 license，并新增聚焦测试。\n",
    encoding="utf-8",
)

summary = {
    "schema_version": "research.rank_ic_objective_validation.v1",
    "status": "completed",
    "source_id": SOURCE_ID,
    "candidate_name": "rank_ic_objective",
    "target_subsystem": "src/objective",
    "target_interface": "src.objective.evaluate_prediction_objectives + src.objective.build_promotion_scores",
    "objective_schema_version": scores["rank_ic_aligned_signal"]["schema_version"],
    "rank_ic_aligned_model": "rank_ic_aligned_signal",
    "rank_ic_aligned_score": scores["rank_ic_aligned_signal"]["rank_ic"],
    "rank_ic_aligned_mse": scores["rank_ic_aligned_signal"]["mse"],
    "mse_aligned_model": "mse_aligned_signal",
    "mse_aligned_score": scores["mse_aligned_signal"]["rank_ic"],
    "mse_aligned_mse": scores["mse_aligned_signal"]["mse"],
    "rank_group_count": scores["rank_ic_aligned_signal"]["rank_group_count"],
    "implementation_allowed": False,
    "promotion_target": "research_only",
    "evidence_requirements": [
        "license_review",
        "research_notebook",
        "focused_pytest",
        "manifest_or_export",
        "chinese_conclusion",
    ],
    "artifacts": {
        "objective_scores": "objective_scores.json",
        "objective_scores_csv": "objective_scores.csv",
        "validation_frame": "rank_ic_validation_frame.csv",
        "source_provenance": "source_provenance.json",
        "license_review": "license_review.md",
    },
}
(CARD / "rank_ic_objective_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
summary


{'schema_version': 'research.rank_ic_objective_validation.v1',
 'status': 'completed',
 'source_id': 'arxiv:2605.00501',
 'candidate_name': 'rank_ic_objective',
 'target_subsystem': 'src/objective',
 'target_interface': 'src.objective.evaluate_prediction_objectives + src.objective.build_promotion_scores',
 'objective_schema_version': 'objective.evaluation.v1',
 'rank_ic_aligned_model': 'rank_ic_aligned_signal',
 'rank_ic_aligned_score': 1.0,
 'rank_ic_aligned_mse': 0.03753262500000001,
 'mse_aligned_model': 'mse_aligned_signal',
 'mse_aligned_score': 0.5,
 'mse_aligned_mse': 5.0416666666666675e-05,
 'rank_group_count': 4,
 'implementation_allowed': False,
 'promotion_target': 'research_only',
 'evidence_requirements': ['license_review',
  'research_notebook',
  'focused_pytest',
  'manifest_or_export',
  'chinese_conclusion'],
 'artifacts': {'objective_scores': 'objective_scores.json',
  'objective_scores_csv': 'objective_scores.csv',
  'validation_frame': 'rank_ic_validation_frame.csv

## 中文结论

- 做了什么：把 `arxiv:2605.00501` / LambdaRankIC 作为 `rank_ic_objective` 研究候选，验证现有 `src.objective.evaluate_prediction_objectives` 能区分 Rank IC 对齐信号和 MSE 标定信号。
- 核心指标：输出 `objective_scores.json/csv`、`rank_ic_validation_frame.csv`、`source_provenance.json`、`license_review.md` 和 `rank_ic_objective_summary.json`；`rank_ic_aligned_signal` 的 rank_ic 高于 `mse_aligned_signal`。
- regime 差异：本 notebook 只验证 objective 评价口径，不做 market regime 切片；后续 regime/window 差异应交给 `src.analysis` 消费 objective score 和 backtest 结果。
- 是否晋升：否。本卡只把外部知识从 backlog 推进到 research evidence，不直接进入 production，也不实现 LambdaRankIC loss。
- 风险：这里只使用 arXiv URL 和自有摘要，未复制论文全文，未 vendoring 代码；如果后续要实现论文 loss，需要单独 license review、聚焦测试和 manifest/export。
- 下一步：在 `knowledge_promotion_audit` 中把该卡作为 `rank_ic_objective` 的 research evidence，然后再决定是否扩展 `src/objective` 的 rank-aware loss 接口。